# Deployment Basics for MLE and Embodied-AI Interviews

This notebook focuses on coding and reasoning skills that commonly appear in deployment interviews:

- FP32, FP16, BF16, and INT8 tradeoffs
- mixed-precision inference
- post-training quantization (PTQ), calibration, and per-channel quantization
- quantization-aware training (QAT) and straight-through estimation
- Conv-BatchNorm fusion
- model memory, latency, throughput, batching, and profiling
- deployment concerns specific to embodied AI

All examples run on CPU with PyTorch. Hardware-specific acceleration depends on the target device and backend.

## 1. Numeric formats

| Format | Bits | Exponent | Mantissa/fraction | Main deployment property |
|---|---:|---:|---:|---|
| FP32 | 32 | 8 | 23 | Baseline accuracy, largest memory traffic |
| FP16 | 16 | 5 | 10 | Good precision, limited dynamic range |
| BF16 | 16 | 8 | 7 | FP32-like range, lower precision |
| INT8 | 8 | n/a | n/a | Small and fast, requires scale/zero-point |

FP16 and BF16 both halve tensor storage versus FP32, but they fail differently. FP16 has more fraction bits but a much smaller exponent range. BF16 usually tolerates large activations better. INT8 can reduce storage by 4x, but only after mapping real values to a discrete integer grid.

In [1]:
import copy
import math
import time
from dataclasses import dataclass
from typing import Iterable, Sequence

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)
torch.set_printoptions(precision=5, sci_mode=False)

for dtype in (torch.float32, torch.float16, torch.bfloat16):
    info = torch.finfo(dtype)
    print(
        f"{str(dtype):14s} bytes={info.bits // 8}, "
        f"smallest_normal={info.tiny:.3e}, max={info.max:.3e}, eps={info.eps:.3e}"
    )

torch.float32  bytes=4, smallest_normal=1.175e-38, max=3.403e+38, eps=1.192e-07
torch.float16  bytes=2, smallest_normal=6.104e-05, max=6.550e+04, eps=9.766e-04
torch.bfloat16 bytes=2, smallest_normal=1.175e-38, max=3.390e+38, eps=7.812e-03


### Interview check: range versus precision

`eps` is the gap between 1 and the next representable value. A smaller `eps` means more precision near 1. `tiny` and `max` describe dynamic range. This explains why BF16 often avoids loss scaling while FP16 training commonly needs it.

Reduced storage does **not** guarantee lower latency. Speedup requires kernels and hardware that efficiently execute the chosen dtype; casts or unsupported operators can erase the benefit.

In [2]:
def relative_error(reference: torch.Tensor, candidate: torch.Tensor) -> float:
    denominator = reference.abs().clamp_min(1e-12)
    return ((reference - candidate).abs() / denominator).mean().item()


values = torch.tensor([1e-8, 1e-4, 1.0, 1_000.0, 100_000.0], dtype=torch.float32)
print("FP32:            ", values)
print("FP16 round-trip: ", values.to(torch.float16).to(torch.float32))
print("BF16 round-trip: ", values.to(torch.bfloat16).to(torch.float32))

# Accumulation is another source of error.
small_updates = torch.full((10_000,), 1e-3)
fp32_sum = small_updates.sum(dtype=torch.float32)
fp16_sum = small_updates.to(torch.float16).sum(dtype=torch.float16).float()
bf16_sum = small_updates.to(torch.bfloat16).sum(dtype=torch.bfloat16).float()
print(f"sums: FP32={fp32_sum.item():.5f}, FP16={fp16_sum.item():.5f}, BF16={bf16_sum.item():.5f}")

FP32:             tensor([    0.00000,     0.00010,     1.00000,  1000.00000, 100000.00000])
FP16 round-trip:  tensor([    0.00000,     0.00010,     1.00000,  1000.00000,         inf])
BF16 round-trip:  tensor([    0.00000,     0.00010,     1.00000,  1000.00000, 99840.00000])
sums: FP32=10.00000, FP16=10.00781, BF16=10.00000


## 2. Model size and mixed-precision inference

A first-order parameter-memory estimate is:

$$\text{parameter bytes} = \sum_i \text{numel}(W_i) \times \text{bytes-per-element}(W_i)$$

Peak inference memory also includes activations, temporary workspaces, runtime overhead, and input/output buffers. For temporal embodied models, cached features or attention KV caches may dominate parameter memory.

In [3]:
class TinyPerceptionNet(nn.Module):
    def __init__(self, num_classes: int = 6):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.head = nn.Linear(32, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.features(x).flatten(1))


def model_bytes(model: nn.Module) -> int:
    tensors = list(model.parameters()) + list(model.buffers())
    return sum(t.numel() * t.element_size() for t in tensors)


model = TinyPerceptionNet().eval()
image = torch.randn(4, 3, 128, 128)
with torch.inference_mode():
    output_fp32 = model(image)

for dtype in (torch.float32, torch.float16, torch.bfloat16):
    converted = copy.deepcopy(model).to(dtype=dtype)
    print(f"{str(dtype):14s}: {model_bytes(converted) / 1024:.1f} KiB")

# Autocast chooses lower precision for eligible operations while preserving
# higher precision for numerically sensitive operations.
with torch.inference_mode(), torch.autocast(device_type="cpu", dtype=torch.bfloat16):
    output_bf16 = model(image)
print("BF16 autocast mean absolute error:", (output_fp32 - output_bf16.float()).abs().mean().item())

torch.float32 : 21.2 KiB
torch.float16 : 10.6 KiB
torch.bfloat16: 10.6 KiB
BF16 autocast mean absolute error: 0.00027984814369119704


## 3. INT8 quantization from first principles

Affine quantization maps a real value $x$ to integer $q$:

$$q = \operatorname{clamp}(\operatorname{round}(x/s) + z, q_{min}, q_{max})$$
$$\hat{x} = s(q-z)$$

- `s` is the positive scale.
- `z` is the integer zero-point.
- symmetric INT8 commonly uses `z = 0` and the range `[-127, 127]`.
- asymmetric quantization can represent skewed ranges more efficiently, especially for non-negative activations.

Weights are commonly quantized **per output channel** because different filters can have very different ranges. Activations are often quantized per tensor because per-channel activation scales can be expensive at runtime.

In [4]:
@dataclass
class QParams:
    scale: torch.Tensor
    zero_point: torch.Tensor
    qmin: int = -127
    qmax: int = 127


def symmetric_qparams(x: torch.Tensor, dims: Sequence[int] | None = None) -> QParams:
    if dims is None:
        max_abs = x.abs().max()
    else:
        max_abs = x.abs().amax(dim=tuple(dims), keepdim=True)
    scale = (max_abs / 127.0).clamp_min(torch.finfo(torch.float32).eps)
    return QParams(scale=scale, zero_point=torch.zeros_like(scale))


def quantize(x: torch.Tensor, params: QParams) -> torch.Tensor:
    q = torch.round(x / params.scale + params.zero_point)
    return q.clamp(params.qmin, params.qmax).to(torch.int8)


def dequantize(q: torch.Tensor, params: QParams) -> torch.Tensor:
    return params.scale * (q.float() - params.zero_point)


x = torch.tensor([-3.2, -1.0, -0.01, 0.0, 0.8, 2.7])
params = symmetric_qparams(x)
x_q = quantize(x, params)
x_hat = dequantize(x_q, params)
print("scale:       ", params.scale.item())
print("quantized:   ", x_q)
print("dequantized: ", x_hat)
print("max error:   ", (x - x_hat).abs().max().item())

scale:        0.025196850299835205
quantized:    tensor([-127,  -40,    0,    0,   32,  107], dtype=torch.int8)
dequantized:  tensor([-3.20000, -1.00787,  0.00000,  0.00000,  0.80630,  2.69606])
max error:    0.009999999776482582


In [5]:
def quantized_linear(
    x: torch.Tensor,
    weight: torch.Tensor,
    bias: torch.Tensor | None = None,
) -> torch.Tensor:
    """Simulate INT8 x INT8 -> INT32 accumulation, then dequantize."""
    x_params = symmetric_qparams(x)
    # One scale for each output row. Reduce over the input-feature dimension.
    w_params = symmetric_qparams(weight, dims=(1,))
    x_q = quantize(x, x_params)
    w_q = quantize(weight, w_params)

    accumulator = x_q.to(torch.int32) @ w_q.to(torch.int32).T
    output = accumulator.float() * x_params.scale * w_params.scale.T
    return output if bias is None else output + bias


batch, in_features, out_features = 8, 64, 12
linear_x = torch.randn(batch, in_features)
linear_w = torch.randn(out_features, in_features) * 0.2
linear_b = torch.randn(out_features) * 0.1
reference = F.linear(linear_x, linear_w, linear_b)
int8_simulated = quantized_linear(linear_x, linear_w, linear_b)

print("mean absolute error:", (reference - int8_simulated).abs().mean().item())
print("mean relative error:", relative_error(reference, int8_simulated))
assert torch.allclose(reference, int8_simulated, atol=0.12, rtol=0.12)

mean absolute error: 0.010648239403963089
mean relative error: 0.027037784457206726


### Why INT32 accumulation?

A dot product sums many INT8 products. The product of two signed INT8 values needs roughly 16 bits, and the sum needs additional headroom. Production kernels therefore usually accumulate into INT32 before rescaling to FP16/FP32 or requantizing to INT8.

The simulation above demonstrates the math, not performance. It repeatedly quantizes tensors and uses generic CPU operators; a real speedup needs packed weights and optimized kernels such as TensorRT, cuDNN, oneDNN, XNNPACK, or a device-specific accelerator runtime.

## 4. Calibration and outliers

PTQ estimates activation ranges from a representative calibration set. A single extreme outlier can make the scale too large, wasting most INT8 levels on a range that typical values never use.

Common observers/calibrators:

- min/max: simple but sensitive to outliers
- moving average min/max: smooths batch variation
- percentile: clips rare extremes
- histogram/KL or MSE search: chooses a clipping threshold that minimizes a distribution-level objective

Calibration data should match deployment data: sensors, weather, lighting, motion, preprocessing, sequence length, and difficult safety-relevant cases.

In [6]:
def qparams_from_threshold(threshold: torch.Tensor) -> QParams:
    scale = (threshold / 127.0).clamp_min(torch.finfo(torch.float32).eps)
    return QParams(scale=scale, zero_point=torch.zeros_like(scale))


def quantization_mse(x: torch.Tensor, params: QParams) -> float:
    reconstructed = dequantize(quantize(x, params), params)
    return F.mse_loss(reconstructed, x).item()


typical = torch.randn(50_000)
with_outliers = torch.cat([typical, torch.tensor([-30.0, 40.0])])

minmax = qparams_from_threshold(with_outliers.abs().max())
p999 = qparams_from_threshold(torch.quantile(with_outliers.abs(), 0.999))

print(f"min/max scale:       {minmax.scale.item():.5f}")
print(f"99.9% scale:         {p999.scale.item():.5f}")
print(f"typical-value MSE, min/max: {quantization_mse(typical, minmax):.7f}")
print(f"typical-value MSE, 99.9%:   {quantization_mse(typical, p999):.7f}")
print("Tradeoff: percentile calibration clips the rare outliers.")

min/max scale:       0.31496
99.9% scale:         0.02567
typical-value MSE, min/max: 0.0082349
typical-value MSE, 99.9%:   0.0001689
Tradeoff: percentile calibration clips the rare outliers.


## 5. Quantization-aware training (QAT)

QAT inserts **fake quantization** into the forward pass: values are quantized and immediately dequantized so the model sees quantization error during training. The round operation has zero gradient almost everywhere, so QAT commonly uses a straight-through estimator (STE): pretend fake quantization was the identity during backpropagation.

A compact STE expression is:

```python
y = x + (fake_quantized_x - x).detach()
```

The forward value is `fake_quantized_x`, while the gradient with respect to `x` is approximately 1.

In [7]:
def fake_quantize_ste(x: torch.Tensor) -> torch.Tensor:
    params = symmetric_qparams(x.detach())
    quantized_value = dequantize(quantize(x.detach(), params), params)
    return x + (quantized_value - x).detach()


class QATLinear(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(out_features, in_features) * 0.1)
        self.bias = nn.Parameter(torch.zeros(out_features))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.linear(fake_quantize_ste(x), fake_quantize_ste(self.weight), self.bias)


qat_layer = QATLinear(4, 2)
qat_input = torch.randn(16, 4)
qat_target = torch.randn(16, 2)
loss = F.mse_loss(qat_layer(qat_input), qat_target)
loss.backward()
print("loss:", loss.item())
print("weight gradient norm:", qat_layer.weight.grad.norm().item())
assert qat_layer.weight.grad.abs().sum() > 0

loss: 1.1590707302093506
weight gradient norm: 0.7106606364250183


## 6. Conv-BatchNorm fusion

At inference, BatchNorm uses fixed running statistics and can be folded into the preceding convolution. For output channel $c$:

$$W'_c = \frac{\gamma_c}{\sqrt{\sigma_c^2 + \epsilon}} W_c$$
$$b'_c = \beta_c + \frac{\gamma_c}{\sqrt{\sigma_c^2 + \epsilon}}(b_c - \mu_c)$$

Fusion removes an operator and memory round trip. It must use an eval-mode BatchNorm with valid running statistics.

In [8]:
def fuse_conv_bn(conv: nn.Conv2d, bn: nn.BatchNorm2d) -> nn.Conv2d:
    if conv.out_channels != bn.num_features:
        raise ValueError("Conv output channels must match BatchNorm features")

    fused = nn.Conv2d(
        conv.in_channels,
        conv.out_channels,
        conv.kernel_size,
        conv.stride,
        conv.padding,
        conv.dilation,
        conv.groups,
        bias=True,
        padding_mode=conv.padding_mode,
    )

    conv_bias = conv.bias if conv.bias is not None else torch.zeros_like(bn.running_mean)
    inv_std = torch.rsqrt(bn.running_var + bn.eps)
    channel_scale = bn.weight * inv_std

    with torch.no_grad():
        fused.weight.copy_(conv.weight * channel_scale[:, None, None, None])
        fused.bias.copy_(bn.bias + (conv_bias - bn.running_mean) * channel_scale)
    return fused


conv = nn.Conv2d(3, 8, kernel_size=3, padding=1, bias=False)
bn = nn.BatchNorm2d(8)
# Give the running statistics nontrivial values, as if training had occurred.
bn.running_mean.copy_(torch.randn(8))
bn.running_var.copy_(torch.rand(8) + 0.5)
conv.eval()
bn.eval()
fused_conv = fuse_conv_bn(conv, bn).eval()

fusion_input = torch.randn(2, 3, 32, 32)
with torch.inference_mode():
    unfused_output = bn(conv(fusion_input))
    fused_output = fused_conv(fusion_input)
print("maximum fusion error:", (unfused_output - fused_output).abs().max().item())
assert torch.allclose(unfused_output, fused_output, atol=1e-5, rtol=1e-5)

maximum fusion error: 1.9073486328125e-06


## 7. Latency, throughput, and correct benchmarking

- **Latency:** time for one request, often reported as p50/p95/p99.
- **Throughput:** samples or frames processed per second.
- **Deadline miss rate:** fraction of requests exceeding the control-loop deadline.
- **Jitter:** variation in latency; important for predictable control.

Benchmark the complete deployed path, including preprocessing, host-device transfer, inference, postprocessing, and synchronization. Warm up kernels, use inference mode, synchronize asynchronous devices, and report distributions rather than only a mean.

In [9]:
def benchmark_ms(
    fn,
    warmup: int = 10,
    iterations: int = 50,
    synchronize=None,
) -> dict[str, float]:
    synchronize = synchronize or (lambda: None)
    for _ in range(warmup):
        fn()
    synchronize()

    samples = []
    for _ in range(iterations):
        synchronize()
        start = time.perf_counter()
        fn()
        synchronize()
        samples.append((time.perf_counter() - start) * 1_000)

    times = torch.tensor(samples)
    return {
        "mean_ms": times.mean().item(),
        "p50_ms": times.quantile(0.50).item(),
        "p95_ms": times.quantile(0.95).item(),
        "p99_ms": times.quantile(0.99).item(),
    }


with torch.inference_mode():
    stats = benchmark_ms(lambda: model(image), warmup=5, iterations=20)
print(stats)
print(f"approximate throughput: {len(image) * 1000 / stats['mean_ms']:.1f} images/s")

{'mean_ms': 1.580933928489685, 'p50_ms': 1.5527794361114502, 'p95_ms': 2.357952117919922, 'p99_ms': 2.6204488277435303}
approximate throughput: 2530.2 images/s


### Batching tradeoff

Batching often improves accelerator utilization and throughput but adds queueing delay and activation memory. A cloud service may use dynamic batching; a robot with a strict 50 ms control period may prefer batch size 1. Always optimize the metric constrained by the product, not an isolated model benchmark.

In [10]:
def pad_point_clouds(point_clouds: Iterable[torch.Tensor]) -> tuple[torch.Tensor, torch.Tensor]:
    """Pad variable-length [N, D] point clouds and return a validity mask."""
    point_clouds = list(point_clouds)
    if not point_clouds:
        raise ValueError("Expected at least one point cloud")
    feature_dim = point_clouds[0].shape[1]
    if any(pc.ndim != 2 or pc.shape[1] != feature_dim for pc in point_clouds):
        raise ValueError("All point clouds must have shape [N, same_feature_dim]")

    max_points = max(pc.shape[0] for pc in point_clouds)
    batch = point_clouds[0].new_zeros((len(point_clouds), max_points, feature_dim))
    valid = torch.zeros((len(point_clouds), max_points), dtype=torch.bool)
    for i, points in enumerate(point_clouds):
        count = points.shape[0]
        batch[i, :count] = points
        valid[i, :count] = True
    return batch, valid


clouds = [torch.randn(5, 4), torch.randn(8, 4), torch.randn(3, 4)]
padded, mask = pad_point_clouds(clouds)
masked_mean = (padded * mask[..., None]).sum(dim=1) / mask.sum(dim=1, keepdim=True)
expected_mean = torch.stack([cloud.mean(dim=0) for cloud in clouds])
print("padded shape:", tuple(padded.shape))
print("valid points:", mask.sum(dim=1).tolist())
assert torch.allclose(masked_mean, expected_mean)

padded shape: (3, 8, 4)
valid points: [5, 8, 3]


## 8. Deployment pipeline and optimization order

A practical workflow:

1. Freeze preprocessing and the model in `eval()` mode.
2. Export a stable graph (`torch.export`, ONNX, or a runtime-native format).
3. Validate numerics against eager FP32 on representative and adversarial inputs.
4. Profile the end-to-end pipeline and identify compute, memory, transfer, and queueing bottlenecks.
5. Apply low-risk graph optimizations: constant folding, dead-code removal, and operator fusion.
6. Try FP16/BF16 where the target supports it.
7. Try INT8 PTQ with representative calibration; use QAT if the quality loss is unacceptable.
8. Tune shapes, batching, memory reuse, and concurrency for the actual deadline.
9. Re-run task metrics, slice metrics, latency percentiles, memory, power, and thermal tests.

Other common techniques include structured pruning, distillation, lower input resolution, early exit, sparse or voxelized perception, efficient attention, caching, and replacing unsupported operators. Unstructured sparsity only speeds up inference when the hardware and kernels exploit its sparsity pattern.

## 9. Embodied-AI deployment checklist

Embodied systems turn latency and numerical errors into physical behavior. Discuss these explicitly in interviews:

| Area | Questions to ask |
|---|---|
| Real-time behavior | What are the control frequency, p99 deadline, jitter limit, and timeout behavior? |
| Sensor timing | Are camera/LiDAR timestamps synchronized? How are stale or dropped frames handled? |
| Temporal state | Is recurrent state reset correctly? Can cached state become inconsistent after packet loss? |
| Shape variability | Are point counts, image sizes, agent counts, and sequence lengths bounded? |
| Safety | Is there a deterministic fallback, health monitor, watchdog, and out-of-distribution response? |
| Quality | Are metrics checked by distance, speed, lighting, weather, object class, and rare-event slice? |
| Hardware | Are memory, power, thermal throttling, startup time, and unsupported operators measured? |
| Operations | Are model/version hashes, telemetry, rollback, staged rollout, and reproducibility available? |

For quantization, compare more than aggregate accuracy. Small perturbations can alter confidence thresholds, NMS decisions, trajectories, occupancy boundaries, or action selection. Evaluate the final task and closed-loop behavior.

## 10. Interview drills

1. Extend `symmetric_qparams` to asymmetric UINT8 quantization with a learned zero-point.
2. Implement group-wise weight quantization and explain the accuracy/runtime tradeoff versus per-tensor and per-channel schemes.
3. Modify calibration to search over clipping percentiles and select the one with minimum reconstruction MSE.
4. Explain why LayerNorm, Softmax, regression heads, and small final layers are sometimes kept in higher precision.
5. Given a 100M-parameter model, estimate weight storage in FP32, FP16/BF16, INT8, and INT4.
6. Design an experiment to determine whether a perception pipeline is compute-bound, memory-bound, transfer-bound, or input-bound.
7. A model averages 20 ms but has p99 latency of 85 ms against a 50 ms deadline. Explain why it is not production-ready and propose measurements and fixes.
8. Design a validation suite for quantizing a camera-LiDAR fusion model used in closed-loop planning.

### Quick memory answer

Ignoring metadata and scales, 100M parameters require approximately 400 MB in FP32, 200 MB in FP16/BF16, 100 MB in INT8, and 50 MB in INT4. Real artifacts also store quantization scales, zero-points, alignment padding, runtime metadata, and sometimes an FP copy.

## Summary

- FP16 offers more precision; BF16 offers more range.
- INT8 quantization is a scale/zero-point mapping plus integer accumulation and rescaling.
- Per-channel weight quantization usually preserves accuracy better than per-tensor weight quantization.
- PTQ quality depends heavily on calibration data and outlier handling; QAT lets weights adapt to quantization noise.
- Fusion, export, batching, and memory movement can matter as much as dtype.
- Deployment success is measured end to end: task quality, p99 latency, deadline misses, memory, power, thermal behavior, and safe fallback.